# E-Commerce API — Setup Test

Runs end-to-end checks against the local server at `http://localhost:8000`.
All cells must pass for setup to be considered complete.

In [ ]:
import requests

BASE = "http://localhost:8000"
passed = []
failed = []

def check(name, condition, detail=""):
    if condition:
        passed.append(name)
        print(f"[OK] {name}")
    else:
        failed.append(name)
        print(f"[FAIL] {name}" + (f": {detail}" if detail else ""))

print(f"Testing server at {BASE}\n")

## 1. Categories

In [ ]:
r = requests.get(f"{BASE}/api/categories")
cats = r.json()

check("GET /api/categories status 200", r.status_code == 200, r.status_code)
check("categories is a list", isinstance(cats, list), type(cats))
check("at least 1 category", len(cats) >= 1, len(cats))
check("categories have 'name' field", all('name' in c for c in cats))
print(f"   Found {len(cats)} categories: {[c['name'] for c in cats[:5]]}{'...' if len(cats) > 5 else ''}")

## 2. Products — list, search, category filter

In [ ]:
r = requests.get(f"{BASE}/api/items")
items = r.json()

check("GET /api/items status 200", r.status_code == 200, r.status_code)
check("items is a list", isinstance(items, list), type(items))
check("at least 1 item", len(items) >= 1, len(items))
print(f"   Total products: {len(items)}")

# Search
r2 = requests.get(f"{BASE}/api/items", params={"q": "egg"})
results = r2.json()
check("search ?q=egg returns results", r2.status_code == 200 and len(results) >= 1, len(results))
print(f"   Search 'egg' -> {len(results)} result(s): {[i.get('name','?') for i in results[:3]]}")

# Category filter
if cats:
    first_cat = cats[0]['name']
    r3 = requests.get(f"{BASE}/api/items", params={"category": first_cat})
    cat_items = r3.json()
    check(f"category filter '{first_cat}' works", r3.status_code == 200 and isinstance(cat_items, list))
    print(f"   Category '{first_cat}' -> {len(cat_items)} item(s)")

## 3. Product Detail

In [ ]:
if items:
    item_id = items[0]['id']
    r = requests.get(f"{BASE}/api/items/{item_id}")
    item = r.json()
    check(f"GET /api/items/{item_id} status 200", r.status_code == 200, r.status_code)
    check("item has name, price, id", all(k in item for k in ['id', 'name', 'price']), list(item.keys()))
    print(f"   Item: {item.get('name')} @ Rs {item.get('price')}")

r404 = requests.get(f"{BASE}/api/items/999999")
check("GET /api/items/999999 returns 404", r404.status_code == 404, r404.status_code)

## 4. Cart — add item, view cart

In [ ]:
if items:
    item_id = items[0]['id']
    
    # Add to cart
    r = requests.post(f"{BASE}/api/cart", json={"item_id": item_id, "quantity": 2})
    check("POST /api/cart status 200", r.status_code == 200, r.status_code)
    
    # View cart
    r2 = requests.get(f"{BASE}/api/cart")
    cart = r2.json()
    check("GET /api/cart status 200", r2.status_code == 200, r2.status_code)
    check("cart has 'items' key", 'items' in cart, list(cart.keys()))
    check("cart has at least 1 item", len(cart.get('items', [])) >= 1)
    check("cart has 'total' key", 'total' in cart, list(cart.keys()))
    print(f"   Cart total: Rs {cart.get('total')}, items: {len(cart.get('items', []))}")

## 5. Place Order

In [ ]:
r = requests.post(f"{BASE}/api/orders")
check("POST /api/orders status 200", r.status_code == 200, r.status_code)

if r.status_code == 200:
    order = r.json()
    check("order has 'order_id'", 'order_id' in order or 'id' in order, list(order.keys()))
    print(f"   Order placed: {order}")

# Cart should be empty now
r2 = requests.get(f"{BASE}/api/cart")
cart_after = r2.json()
check("cart empty after order", len(cart_after.get('items', [])) == 0, cart_after)

# Placing order on empty cart should fail
r3 = requests.post(f"{BASE}/api/orders")
check("POST /api/orders on empty cart returns 4xx", r3.status_code >= 400, r3.status_code)

## 6. Order History

In [ ]:
r = requests.get(f"{BASE}/api/orders")
orders = r.json()

check("GET /api/orders status 200", r.status_code == 200, r.status_code)
check("orders is a list", isinstance(orders, list), type(orders))
check("at least 1 order in history", len(orders) >= 1, len(orders))
print(f"   Total orders: {len(orders)}")

## Results

In [ ]:
total = len(passed) + len(failed)
print("=" * 40)
print(f"  Results: {len(passed)}/{total} passed")
print("=" * 40)

if failed:
    print("\nFailed checks:")
    for f in failed:
        print(f"  [FAIL] {f}")
    raise AssertionError(f"{len(failed)} check(s) failed: {failed}")
else:
    print("\nAll checks passed. Server is ready!")